# 11 — Caruana Ensemble Selection Prep

**Reporting/assembly only -- nothing here retrains anything.** Every input this
notebook needs (per-config, per-(repeat, fold) out-of-fold predictions for all 11
real configs) already exists on disk from `05_cv_comparison`; this notebook only
loads those files, joins in ground truth, and saves per-isoform OOF matrices for
`11b_caruana_selection.ipynb` to consume.

**Provenance note.** This notebook did not exist before this session, despite
CLAUDE.md's (uncommitted) notebook-status log describing notebook 11's "prep" step
as already "built." That line did not match the actual repository state -- confirmed
via `git log --all` (including deleted files, all branches), a full filesystem
search, and a check of this session's other working directories -- so it was not
loaded from disk here; it is built fresh, from real, already-computed CV
predictions, per CLAUDE.md's "never fetch, invent, or approximate data" rule.
Flagged to and confirmed with the user before proceeding (2026-09-03).

**Design decision, flagged rather than picked silently.** The task that motivated
this notebook described the target as "one (n_compounds x 11) OOF matrix per
isoform." Taken completely literally, that would mean one row per unique compound
-- but this project's CV is a *repeated* 5x5 design (5 independent 5-fold
partitions, `cv_folds.csv`, notebook 03), so a single compound has 5 different OOF
predictions per config (one per repeat), each from a different training split. This
notebook pools **all 25 (repeat, fold) OOF predictions** per isoform (5 rows per
compound, not 1), matching notebook 08's own working granularity exactly (08's
`build_pool_wide` is one row per `(repeat, fold, inchikey)`) rather than arbitrarily
discarding 4 of the 5 repeats to hit a literal `n_compounds` row count. This gives
Caruana's hillclimbing/bagging procedure a larger, richer OOF set (consistent with
the paper's own preference for larger hillclimbing sets) and keeps this notebook's
output directly comparable, fold-for-fold, to 08's saved per-fold ST-RAE scores --
which the paired comparison in 11b needs. Every compound still contributes exactly
5 rows, not a variable number, so no compound is systematically over- or
under-weighted relative to any other.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # so `from src... import ...` works regardless of cwd

import importlib.metadata

import pandas as pd

print(f"python: {sys.version.split()[0]}")
for pkg in ["numpy", "pandas"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")

OUT05 = REPO_ROOT / "outputs" / "05_cv_comparison"
OUT = REPO_ROOT / "outputs" / "11_caruana_prep"
OUT.mkdir(parents=True, exist_ok=True)

CURATED_PATH = REPO_ROOT / "data" / "processed" / "train_inhibition_curated.csv"
curated = pd.read_csv(CURATED_PATH)
print(f"curated: {curated.shape}, unique inchikeys: {curated['inchikey'].nunique()}")

PRED_DIR_05 = OUT05 / "predictions"
N_REPEATS, N_FOLDS = 5, 5
ISOFORMS = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]

# Derived from 05's own leaderboard (not hardcoded from memory) so a config renamed
# or dropped since 05 was written cannot pass unnoticed into this pool.
leaderboard_1a2 = pd.read_csv(OUT05 / "leaderboard_CYP1A2.csv")
ALL_11_CONFIGS = sorted(leaderboard_1a2.loc[leaderboard_1a2["config"] != "naive_mean", "config"].tolist())
print(f"\ncandidate library ({len(ALL_11_CONFIGS)} configs): {ALL_11_CONFIGS}")
assert len(ALL_11_CONFIGS) == 11, f"expected exactly 11 real configs from 05, got {len(ALL_11_CONFIGS)}"


python: 3.11.13
numpy: 1.26.4
pandas: 2.3.3
curated: (4905, 20), unique inchikeys: 4905

candidate library (11 configs): ['chemeleon__lightgbm', 'chemeleon__rf', 'chemeleon__xgboost', 'chemprop_chemeleoninit', 'chemprop_randominit', 'ecfp4_narrow__lightgbm', 'ecfp4_narrow__rf', 'ecfp4_narrow__xgboost', 'mordred_pca__lightgbm', 'mordred_pca__rf', 'mordred_pca__xgboost']


**What this shows:** the curated dataset (4,905 compounds) and 05's own
11-real-config leaderboard both load cleanly, and the candidate library derived
from that leaderboard (excluding the `naive_mean` baseline row) is exactly the 11
configs CLAUDE.md's notebook-11 entry describes ("9 tabular incl. XGBoost, 2
chemprop") -- asserted, not assumed.

In [2]:
def load_model_oof(tag: str, isoform_col: str) -> pd.DataFrame:
    """One config's 25 (repeat, fold) OOF prediction files for one isoform column,
    concatenated with repeat/fold columns attached. Same source directory and
    file-naming convention as notebook 08's `load_model_oof`."""
    rows = []
    for repeat in range(N_REPEATS):
        for fold in range(N_FOLDS):
            path = PRED_DIR_05 / f"{tag}__repeat{repeat}_fold{fold}.csv"
            df = pd.read_csv(path, usecols=["Molecule_Name", "inchikey", isoform_col])
            df["repeat"], df["fold"] = repeat, fold
            rows.append(df)
    return pd.concat(rows, ignore_index=True)


def build_oof_long(isoform: str) -> pd.DataFrame:
    """One row per (repeat, fold, inchikey) *that has a real ground-truth label for
    this isoform*, one column per one of the 11 configs' OOF prediction, plus that
    isoform's true value / conf_low / conf_high from the curated data (constant
    across configs -- a property of the assay measurement, not of any model).

    The multitask models predict every isoform for every compound in a fold's test
    split regardless of whether that compound was actually assayed against that
    isoform -- confirmed directly (0 NaNs in any raw prediction file for any
    isoform). Rows without a real ground-truth label for this isoform are dropped
    here (same rule `score_activity_predictions` itself applies: "each endpoint is
    scored only on the compounds that have a ground-truth value for that
    endpoint") -- logged explicitly below, not a silent drop.
    """
    isoform_col = f"{isoform}_pIC50_direct_inhibition"
    conf_low_col, conf_high_col = f"{isoform_col}_conf_low", f"{isoform_col}_conf_high"

    wide, n_first = None, None
    for tag in ALL_11_CONFIGS:
        long_df = load_model_oof(tag, isoform_col).rename(columns={isoform_col: tag})
        print(f"  loaded {tag}: {len(long_df)} rows ({N_REPEATS * N_FOLDS} folds, all compounds)")
        if wide is None:
            wide = long_df[["repeat", "fold", "inchikey", "Molecule_Name", tag]]
            n_first = len(wide)
        else:
            wide = wide.merge(
                long_df[["repeat", "fold", "inchikey", tag]], on=["repeat", "fold", "inchikey"], how="inner"
            )
    print(f"  pooled table after inner-join across all {len(ALL_11_CONFIGS)} configs: {len(wide)} rows (first config alone: {n_first})")
    assert len(wide) == n_first, f"{isoform}: inner join across the 11 configs dropped rows -- configs do not share an identical predicted-compound set per fold"
    assert wide.shape[1] - 4 == 11, f"{isoform}: expected exactly 11 config columns after the join, got {wide.shape[1] - 4}"

    ground_truth = curated[["inchikey", isoform_col, conf_low_col, conf_high_col]].rename(
        columns={isoform_col: "y_true", conf_low_col: "conf_low", conf_high_col: "conf_high"}
    )
    n_before = len(wide)
    merged = wide.merge(ground_truth, on="inchikey", how="left")
    assert len(merged) == n_before, f"{isoform}: joining ground truth changed row count -- duplicate inchikeys in curated data?"

    has_label = merged["y_true"].notna()
    n_untested = (~has_label).sum()
    print(f"  rows before ground-truth filter: {len(merged)}; rows with no {isoform} label (compound not assayed for this isoform, expected -- dropped): {n_untested}")
    merged = merged.loc[has_label].reset_index(drop=True)
    print(f"  rows after ground-truth filter: {len(merged)}")

    check_cols = ["y_true", "conf_low", "conf_high"] + ALL_11_CONFIGS
    n_nan = merged[check_cols].isna().sum()
    if n_nan.any():
        raise ValueError(
            f"{isoform}: unexpected NaNs remain after the ground-truth filter (these columns should now be complete):\n{n_nan[n_nan > 0]}"
        )
    print(f"  NaN check passed: 0 unexpected NaNs across true/conf_low/conf_high/{len(ALL_11_CONFIGS)} config columns")

    return merged


**What this shows:** the shared assembly pipeline -- load all 11 configs'
25-fold OOF predictions, inner-join them on `(repeat, fold, inchikey)` (asserted
lossless -- all 11 configs predict the same compound set per fold, since they're
all evaluated on the same frozen `cv_folds.csv` splits), then attach ground truth
and drop rows for compounds not actually assayed against that isoform (expected,
logged, not an error). Nothing has run yet for any isoform -- the loop below runs
it for all four.

In [3]:
oof_tables = {}
for isoform in ISOFORMS:
    print(f"=== {isoform} ===")
    oof_tables[isoform] = build_oof_long(isoform)
    n_unique = oof_tables[isoform]["inchikey"].nunique()
    n_expected = n_unique * N_REPEATS
    print(f"  {n_unique} unique compounds x {N_REPEATS} repeats = {n_expected} expected rows (got {len(oof_tables[isoform])})")
    assert len(oof_tables[isoform]) == n_expected, f"{isoform}: row count does not match n_unique_compounds x n_repeats -- a compound is missing from some repeat's fold assignment"

    curated_n_valid = curated[f"{isoform}_pIC50_direct_inhibition"].notna().sum()
    assert n_unique == curated_n_valid, f"{isoform}: {n_unique} unique OOF compounds != {curated_n_valid} curated valid-label compounds"

    path = OUT / f"oof_long_{isoform}.csv"
    oof_tables[isoform].to_csv(path, index=False)
    print(f"  wrote {path}: {oof_tables[isoform].shape}\n")


=== CYP1A2 ===
  loaded chemeleon__lightgbm: 24525 rows (25 folds, all compounds)
  loaded chemeleon__rf: 24525 rows (25 folds, all compounds)
  loaded chemeleon__xgboost: 24525 rows (25 folds, all compounds)
  loaded chemprop_chemeleoninit: 24525 rows (25 folds, all compounds)


  loaded chemprop_randominit: 24525 rows (25 folds, all compounds)


  loaded ecfp4_narrow__lightgbm: 24525 rows (25 folds, all compounds)
  loaded ecfp4_narrow__rf: 24525 rows (25 folds, all compounds)
  loaded ecfp4_narrow__xgboost: 24525 rows (25 folds, all compounds)
  loaded mordred_pca__lightgbm: 24525 rows (25 folds, all compounds)
  loaded mordred_pca__rf: 24525 rows (25 folds, all compounds)


  loaded mordred_pca__xgboost: 24525 rows (25 folds, all compounds)
  pooled table after inner-join across all 11 configs: 24525 rows (first config alone: 24525)
  rows before ground-truth filter: 24525; rows with no CYP1A2 label (compound not assayed for this isoform, expected -- dropped): 17465
  rows after ground-truth filter: 7060
  NaN check passed: 0 unexpected NaNs across true/conf_low/conf_high/11 config columns
  1412 unique compounds x 5 repeats = 7060 expected rows (got 7060)


  wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/11_caruana_prep/oof_long_CYP1A2.csv: (7060, 18)

=== CYP2C9 ===
  loaded chemeleon__lightgbm: 24525 rows (25 folds, all compounds)
  loaded chemeleon__rf: 24525 rows (25 folds, all compounds)
  loaded chemeleon__xgboost: 24525 rows (25 folds, all compounds)
  loaded chemprop_chemeleoninit: 24525 rows (25 folds, all compounds)
  loaded chemprop_randominit: 24525 rows (25 folds, all compounds)


...

  loaded mordred_pca__rf: 24525 rows (25 folds, all compounds)
  loaded mordred_pca__xgboost: 24525 rows (25 folds, all compounds)
  pooled table after inner-join across all 11 configs: 24525 rows (first config alone: 24525)
  rows before ground-truth filter: 24525; rows with no CYP2D6 label (compound not assayed for this isoform, expected -- dropped): 17060
  rows after ground-truth filter: 7465
  NaN check passed: 0 unexpected NaNs across true/conf_low/conf_high/11 config columns
  1493 unique compounds x 5 repeats = 7465 expected rows (got 7465)
  wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/11_caruana_prep/oof_long_CYP2D6.csv: (7465, 18)

=== CYP3A4 ===
  loaded chemeleon__lightgbm: 24525 rows (25 folds, all compounds)
  loaded chemeleon__rf: 24525 rows (25 folds, all compounds)
  loaded chemeleon__xgboost: 24525 rows (25 folds, all compounds)


  loaded chemprop_chemeleoninit: 24525 rows (25 folds, all compounds)


  loaded chemprop_randominit: 24525 rows (25 folds, all compounds)
  loaded ecfp4_narrow__lightgbm: 24525 rows (25 folds, all compounds)
  loaded ecfp4_narrow__rf: 24525 rows (25 folds, all compounds)
  loaded ecfp4_narrow__xgboost: 24525 rows (25 folds, all compounds)
  loaded mordred_pca__lightgbm: 24525 rows (25 folds, all compounds)
  loaded mordred_pca__rf: 24525 rows (25 folds, all compounds)
  loaded mordred_pca__xgboost: 24525 rows (25 folds, all compounds)


  pooled table after inner-join across all 11 configs: 24525 rows (first config alone: 24525)
  rows before ground-truth filter: 24525; rows with no CYP3A4 label (compound not assayed for this isoform, expected -- dropped): 12850
  rows after ground-truth filter: 11675
  NaN check passed: 0 unexpected NaNs across true/conf_low/conf_high/11 config columns
  2335 unique compounds x 5 repeats = 11675 expected rows (got 11675)


  wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/11_caruana_prep/oof_long_CYP3A4.csv: (11675, 18)



**What this shows:** every isoform assembled cleanly with the expected row
count (`n_unique_compounds x 5 repeats`, matching the curated dataset's own
valid-label count for that isoform exactly), zero unexpected NaNs, and exactly 11
config columns -- the three conditions the downstream task explicitly said to stop
and flag on if violated. None were. Per-isoform compound counts (1,412 / 1,285 /
1,493 / 2,335 for CYP1A2 / CYP2C9 / CYP2D6 / CYP3A4) match this project's known
per-isoform label coverage from earlier notebooks.

In [4]:
manifest_rows = []
for isoform in ISOFORMS:
    df = oof_tables[isoform]
    manifest_rows.append({
        "isoform": isoform,
        "n_rows": len(df),
        "n_unique_compounds": df["inchikey"].nunique(),
        "n_repeats": N_REPEATS,
        "n_folds": N_FOLDS,
        "n_configs": len(ALL_11_CONFIGS),
        "configs": "|".join(ALL_11_CONFIGS),
    })
manifest = pd.DataFrame(manifest_rows)
print(manifest.drop(columns="configs").to_string(index=False))

manifest_path = OUT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"\nwrote {manifest_path}")


isoform  n_rows  n_unique_compounds  n_repeats  n_folds  n_configs
 CYP1A2    7060                1412          5        5         11
 CYP2C9    6425                1285          5        5         11
 CYP2D6    7465                1493          5        5         11
 CYP3A4   11675                2335          5        5         11

wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/11_caruana_prep/manifest.csv


**What this shows:** `outputs/11_caruana_prep/manifest.csv` records exactly
what was built -- one `oof_long_{isoform}.csv` per isoform (columns: `repeat`,
`fold`, `inchikey`, `Molecule_Name`, one column per one of the 11 configs, `y_true`,
`conf_low`, `conf_high`), plus this manifest confirming row/compound/config counts.
This is what `11b_caruana_selection.ipynb` loads directly -- no retraining, no
test-set data, matching this notebook's own scope.